In [25]:
import csv
import datetime as dt
from typing import Dict, List, Optional, Tuple, Union

import gpflow
import numpy as np
import pandas as pd
import tensorflow as tf
from gpflow.kernels import ChangePoints, Matern32
from sklearn.preprocessing import StandardScaler
from tensorflow_probability import bijectors as tfb

Kernel = gpflow.kernels.base.Kernel

MAX_ITERATIONS = 200

In [26]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from data.pull_data import pull_lobster_data

def calc_returns(srs: pd.Series, offset: int = 1) -> pd.Series:
    """Calculates returns over the specified number of seconds."""
    return srs / srs.shift(offset) - 1.0

In [27]:
df = pull_lobster_data('AAPL')
df['second_returns'] = calc_returns(df['mid_price'], offset=1)

df.head()

,mid_price,second_returns
date,,
2024-01-02 09:30:00,187.150,NaN
2024-01-02 09:30:01,187.100,-0.000267
2024-01-02 09:30:02,186.895,-0.001096
2024-01-02 09:30:03,186.865,-0.000161
2024-01-02 09:30:04,186.965,0.000535


In [28]:
VOL_THRESHOLD = 5
VOL_TARGET = 0.15
SECONDS_PER_MINUTE = 60
SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY = 23400
SECONDS_PER_YEAR = SECONDS_PER_DAY * 252
HALFLIFE_WINSORISE = SECONDS_PER_DAY * 60

#### **5-minute LBW**

In [29]:
data = df.copy()

csv_fields = ["date", "t", "cp_location", "cp_location_norm", "cp_score"]
with open('test.csv', "w") as f:
    writer = csv.writer(f)
    writer.writerow(csv_fields)

In [ ]:
# data['date'] = data.index
data = data.reset_index()

lookback_window_length = 5 * SECONDS_PER_MINUTE

for window_end in range(lookback_window_length + 1, len(data)):
    tmp = data.iloc[window_end - (lookback_window_length + 1):window_end][['date', 'second_returns']].copy()
    tmp['X'] = tmp.index.astype(float)
    tmp = tmp.rename(columns={'second_returns': 'Y'})
    time_index = window_end - 1
    window_date = tmp['date'].iloc[-1].strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if use_kM_hyp_to_initialise_kC:
            cp_score, cp_loc, cp_loc_normalised, _, _ = changepoint_loc_and_score(
                ts_data_window,
            )
        else:
            cp_score, cp_loc, cp_loc_normalised, _, _ = changepoint_loc_and_score(
                ts_data_window,
                k1_lengthscale=1.0,
                k1_variance=1.0,
                k2_lengthscale=1.0,
                k2_variance=1.0,
                kC_likelihood_variance=1.0,
            )

    except:
        # write as NA when fails and will deal with this later
        cp_score, cp_loc, cp_loc_normalised = "NA", "NA", "NA"

tmp

,date,Y,X
0,2024-01-02 09:30:00,NaN,0.0
1,2024-01-02 09:30:01,-0.000267,1.0
2,2024-01-02 09:30:02,-0.001096,2.0
3,2024-01-02 09:30:03,-0.000161,3.0
4,2024-01-02 09:30:04,0.000535,4.0
...,...,...,...
296,2024-01-02 09:34:56,0.000160,296.0
297,2024-01-02 09:34:57,-0.000107,297.0
298,2024-01-02 09:34:58,-0.000027,298.0
299,2024-01-02 09:34:59,0.000000,299.0


In [ ]:
# cpd.run_module(
#         data, lookback_window_length, output_file_path, start_date, end_date, USE_KM_HYP_TO_INITIALISE_KC
# )

In [2]:
def run_module(
    time_series_data: pd.DataFrame,
    lookback_window_length: int,
    output_csv_file_path: str,
    start_date: dt.datetime = None,
    end_date: dt.datetime = None,
    use_kM_hyp_to_initialise_kC=True,
):
    """Run the changepoint detection module as described in https://arxiv.org/pdf/2105.13727.pdf
    for all times (in date range if specified). Outputs results to a csv.

    Args:
        time_series_data (pd.DataFrame): time series with date as index and with column daily_returns
        lookback_window_length (int): lookback window length
        output_csv_file_path (str): dull path, including csv extension to output results
        start_date (dt.datetime, optional): start date for module, if None use all (with burnin in period qualt to length of LBW). Defaults to None.
        end_date (dt.datetime, optional): end date for module. Defaults to None.
        use_kM_hyp_to_initialise_kC (bool, optional): initialise Changepoint kernel parameters using the paremters from fitting Matern 3/2 kernel. Defaults to True.
    """
    if start_date and end_date:
        first_window = time_series_data.loc[:start_date].iloc[
            -(lookback_window_length + 1) :, :
        ]
        remaining_data = time_series_data.loc[start_date:end_date, :]
        if remaining_data.index[0] == start_date:
            remaining_data = remaining_data.iloc[1:, :]
        else:
            first_window = first_window.iloc[1:]
        time_series_data = pd.concat([first_window, remaining_data]).copy()
    elif not start_date and not end_date:
        time_series_data = time_series_data.copy()
    elif not start_date:
        time_series_data = time_series_data.iloc[:end_date, :].copy()
    elif not end_date:
        first_window = time_series_data.loc[:start_date].iloc[
            -(lookback_window_length + 1) :, :
        ]
        remaining_data = time_series_data.loc[start_date:, :]
        if remaining_data.index[0] == start_date:
            remaining_data = remaining_data.iloc[1:, :]
        else:
            first_window = first_window.iloc[1:]
        time_series_data = pd.concat([first_window, remaining_data]).copy()

    csv_fields = ["date", "t", "cp_location", "cp_location_norm", "cp_score"]
    with open(output_csv_file_path, "w") as f:
        writer = csv.writer(f)
        writer.writerow(csv_fields)

    time_series_data["date"] = time_series_data.index
    time_series_data = time_series_data.reset_index(drop=True)
    for window_end in range(lookback_window_length + 1, len(time_series_data)):
        ts_data_window = time_series_data.iloc[
            window_end - (lookback_window_length + 1) : window_end
        ][["date", "daily_returns"]].copy()
        ts_data_window["X"] = ts_data_window.index.astype(float)
        ts_data_window = ts_data_window.rename(columns={"daily_returns": "Y"})
        time_index = window_end - 1
        window_date = ts_data_window["date"].iloc[-1].strftime("%Y-%m-%d")

        try:
            if use_kM_hyp_to_initialise_kC:
                cp_score, cp_loc, cp_loc_normalised, _, _ = changepoint_loc_and_score(
                    ts_data_window,
                )
            else:
                cp_score, cp_loc, cp_loc_normalised, _, _ = changepoint_loc_and_score(
                    ts_data_window,
                    k1_lengthscale=1.0,
                    k1_variance=1.0,
                    k2_lengthscale=1.0,
                    k2_variance=1.0,
                    kC_likelihood_variance=1.0,
                )

        except:
            # write as NA when fails and will deal with this later
            cp_score, cp_loc, cp_loc_normalised = "NA", "NA", "NA"

        # #write the reults to the csv
        with open(output_csv_file_path, "a") as f:
            writer = csv.writer(f)
            writer.writerow(
                [window_date, time_index, cp_loc, cp_loc_normalised, cp_score]
            )
